In [53]:
import pandas as pd
import numpy as np
from rdkit import Chem

##################
### Load table ###
##################

profiled_compounds = 'standardized_smiles_profiled.csv'
df = pd.read_csv(profiled_compounds)

#########################################
### Screen for structural liabilities ###
#########################################

def structural_filter():
    
    conditions = (
        (df['PAINS'] == True) |
        (df['BRENK'] == True) |
        (df['NIH'] == True) |
        (df['SA_score'] > 6.0) |
        (df['MW'] > 500.0) |
        (df['TPSA'] > 140.0) |
        (df['HBD'] > 5) |
        (df['HBA'] > 10) |
        (df["Aromatic_Ring_Number"] > 7) |
        (df['RotB'] > 10) |
        (df['LogP'] > 5) |
        (df['LogP'] < 0)
    )
        
    df['Structural_flag'] = conditions
    
    return df

####################################
### Screen for ADMET liabilities ###
####################################

def admet_filter():
    
    conditions = (
        (df['hERG'] > 0.6) |
        (df['Caco2_Wang'] > 6.0) |
        (df['Clearance_Hepatocyte_AZ'] > 15) |
        (df['Clearance_Microsome_AZ'] > 50) |
        (df['PAMPA_NCATS'] < 0.5) |
        (df['PPBR_AZ'] > 90) |
        (df['Solubility_AqSolDB'] < -4.0)
    )
        
    df['ADMET_flag'] = conditions
    
    return df
    
###########################################################
### Run functions and save results as csv and sdf files ###
###########################################################

def combined_filter():
    
    structural_filter()
    admet_filter()

    return df
    
df_filtered = combined_filter()
df_filtered_select = df_filtered[(df_filtered['Structural_flag'] == False) & (df_filtered['ADMET_flag'] == False)]

### Full table of profiled and filtered compounds ###
full_table = 'standardized_smiles_filtered.csv'
df_filtered.to_csv(full_table, index=False)

### Table of select profiled and filtered compounds for docking ###
select_table = 'standardized_smiles_filtered_select.csv'
df_filtered_select.to_csv(select_table, index=False)

### Save select compounds as SDF file ###
# Create SDF writer
writer = Chem.SDWriter("filtered_compounds.sdf")

# Loop through SMILES
for smiles in df_filtered_select["Compound"]:
    mol = Chem.MolFromSmiles(smiles)

    if mol is not None:
        # Store the SMILES as a property
        mol.SetProp("SMILES", smiles)
        writer.write(mol)

writer.close()

print(f'The results have been successfully saved as {full_table} and {select_table}!')
df_filtered_select

The results have been successfully saved as standardized_smiles_filtered.csv and standardized_smiles_filtered_select.csv!


,Compound,MW,LogP,HBD,HBA,Aromatic_Ring_Number,TPSA,RotB,QED,Formal_Charge,...,Solubility_AqSolDB,hERG,Caco2_Wang,BBB_Martins,Clearance_Hepatocyte_AZ,Clearance_Microsome_AZ,PAMPA_NCATS,PPBR_AZ,Structural_flag,ADMET_flag
59,O=C(NS(=O)(=O)C(F)F)C(Cc1cccnc1)C1CCCC1,332.37,2.10,1,4,1,76.13,6,0.87,0,...,-2.39,0.35,-4.93,0.82,1.80,0.11,0.86,75.06,False,False
80,CC(C)(C(=O)NC1CC(S(C)(=O)=O)C1)c1cncs1,302.42,1.11,1,5,1,76.13,4,0.90,0,...,-1.19,0.11,-4.73,0.76,14.09,1.25,0.86,32.66,False,False
92,CC1=CC(C(=O)NCCC(=O)C2CCCCC2)=NS(=O)(=O)N1,327.41,0.83,2,4,0,104.70,5,0.78,0,...,-2.99,0.04,-5.27,0.78,8.06,21.59,0.78,78.21,False,False
107,CS(=O)(=O)CNC(=O)Nc1nc(C2CCCCC2)ns1,318.42,1.71,2,6,1,101.05,4,0.88,0,...,-3.07,0.41,-5.05,0.87,12.30,18.88,0.92,85.03,False,False
141,Cn1nc(Br)c(Sc2cc(Br)n[nH]c2=O)n1,367.03,1.57,1,5,2,76.46,2,0.87,0,...,-3.01,0.02,-5.09,0.87,14.01,6.33,0.92,87.76,False,False
342,Cc1nnsc1CNC(=O)C1CC(=O)NC2CCCCC21,308.41,1.16,2,5,1,83.98,3,0.88,0,...,-1.55,0.14,-5.14,0.93,9.91,2.44,0.88,42.66,False,False
349,Cn1nnc(Cn2cc(C(O)c3ccc(Cl)cc3)nn2)n1,305.73,0.58,1,6,3,94.54,4,0.76,0,...,-2.68,0.03,-5.67,0.69,13.23,5.55,0.56,73.85,False,False
364,CS(=O)(=O)N=S1(=O)CCN(Cc2nccn2-c2ccccc2Cl)CC1,402.93,1.77,0,5,2,84.63,4,0.78,0,...,-1.80,0.52,-5.28,0.85,0.76,1.67,0.77,71.63,False,False
382,CS(=O)(=NC(=O)c1cc(CN)[nH]n1)c1ccc(Br)cn1,358.22,1.32,2,5,2,114.09,3,0.86,0,...,-2.65,0.04,-5.43,0.75,11.11,-2.67,0.80,73.73,False,False
384,COc1cnccc1NC(=O)NCC1(S(N)(=O)=O)CC1,300.34,0.03,3,5,1,123.41,5,0.71,0,...,-1.58,0.15,-5.31,0.84,14.45,16.55,0.75,38.14,False,False


In [51]:
df_filtered_select['Compound']

59                O=C(NS(=O)(=O)C(F)F)C(Cc1cccnc1)C1CCCC1
80                 CC(C)(C(=O)NC1CC(S(C)(=O)=O)C1)c1cncs1
92             CC1=CC(C(=O)NCCC(=O)C2CCCCC2)=NS(=O)(=O)N1
107                   CS(=O)(=O)CNC(=O)Nc1nc(C2CCCCC2)ns1
141                      Cn1nc(Br)c(Sc2cc(Br)n[nH]c2=O)n1
342                     Cc1nnsc1CNC(=O)C1CC(=O)NC2CCCCC21
349                  Cn1nnc(Cn2cc(C(O)c3ccc(Cl)cc3)nn2)n1
364         CS(=O)(=O)N=S1(=O)CCN(Cc2nccn2-c2ccccc2Cl)CC1
382             CS(=O)(=NC(=O)c1cc(CN)[nH]n1)c1ccc(Br)cn1
384                   COc1cnccc1NC(=O)NCC1(S(N)(=O)=O)CC1
456                      Cn1nc(Br)nc1NCc1ncc(C(F)(F)F)cn1
463     CN(C)S(=O)(=O)CC(NC(=O)NCCc1nnc2n1CCCCC2)C(F)(F)F
497                O=C1NC(C(=O)NC2COCCOC2)=CC(c2ccccc2)N1
560     COC(=O)N=S1(=O)CCN(C(=O)Nc2ccc(SCCN3CCCC3)cc2)CC1
562                    CS(=O)(=O)NC(=O)Nc1ccccc1NC1CCCCC1
637                         CN(Cc1ncccn1)CC(O)c1cc(Br)no1
638         Cc1cc(CS(=O)(=O)NC(=O)CC(C)NC(=O)C2CCCCC2)no1
676          N